# Checkpoint de entrega — treino final (política P1)

Roda **um** ajuste com TODAS as pessoas do MINDS, política `--politica-final ultima` (sem avaliação sobreposta ao treino — só a última época, nunca a melhor numa validação que também treinou), semente e backbone fixados conforme [`docs/politica-modelo-final-2026-09-14.md`](../../docs/politica-modelo-final-2026-09-14.md) — não escolhidos por desempenho.

**Não é** LOSO, não gera relatório de acurácia, não é avaliação independente. O checkpoint resultante (`modelo_final.pt`) ainda precisa de exportação e paridade próprias — não herda a aprovação do piloto M01.


## 1. Ambiente e código

Antes de executar, **commitar/publicar as correções e informar o SHA completo aprovado** em `LIBRAS_COMMIT_FINAL` (variável de ambiente) ou `COMMIT_APROVADO` abaixo. Não usar uma branch nem preencher com o HEAD antigo: o snapshot deve conter as novas guardas. O notebook para se o SHA não for informado. Nenhum push/merge é feito automaticamente.

Localmente, usar a variável de ambiente para não modificar este notebook rastreado; no Kaggle, o notebook enviado ao editor é separado do clone. Esta receita exige GPU CUDA. O registro de ambiente permite rastrear a execução, não garante determinismo entre versões/dispositivos.

In [ ]:
import os, pathlib, re, subprocess, sys, tarfile, tempfile, json

EM_KAGGLE = pathlib.Path("/kaggle/working").is_dir() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else ".").resolve()
URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
# Preencher com o SHA COMPLETO do commit que inclui estas correções, após
# commit/publicação. Não usar uma branch móvel nem inferir HEAD automaticamente.
COMMIT_APROVADO = os.environ.get("LIBRAS_COMMIT_FINAL", "")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT_APROVADO):
    raise RuntimeError("Defina COMMIT_APROVADO (ou LIBRAS_COMMIT_FINAL) com o SHA completo já publicado.")

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except subprocess.CalledProcessError as erro:
        raise RuntimeError(f"Git falhou: {erro.stderr.strip()}. No Kaggle, habilite Internet e "
                           "confirme que o commit aprovado foi publicado.") from erro

if EM_KAGGLE:
    REPO = BASE / f"libras-final-{COMMIT_APROVADO[:12]}"
    if not REPO.exists():
        git("clone", "--no-checkout", URL, str(REPO))
        git("fetch", "origin", COMMIT_APROVADO, repo=REPO)
        git("checkout", "--detach", COMMIT_APROVADO, repo=REPO)
    # Clone existente nunca recebe reset/merge/checkout implícito.
else:
    REPO = next((p for p in (BASE, *BASE.parents)
                 if (p / "computer-vision-model" / "treino").is_dir()), None)
    if REPO is None:
        raise RuntimeError("Execute localmente dentro do repositório.")

TREINO = (REPO / "computer-vision-model" / "treino").resolve()
if git("rev-parse", "HEAD", repo=REPO) != COMMIT_APROVADO:
    raise RuntimeError("Clone não está no commit aprovado; use um destino novo.")
for nome in ("codigo_final.py", "entrada_final.py", "test_politica_final.py"):
    if not (TREINO / nome).is_file():
        raise RuntimeError(f"Snapshot sem {nome}; publique as correções antes da run.")
sys.path.insert(0, str(TREINO))
import codigo_final
COMMIT_CODIGO = codigo_final.conferir_codigo(REPO, COMMIT_APROVADO)
print("ambiente:", "Kaggle" if EM_KAGGLE else "local")
print("código fixado:", COMMIT_CODIGO, "|", TREINO)

In [ ]:
import importlib.util
faltantes = [pacote for modulo, pacote in (("yaml", "pyyaml"), ("scipy", "scipy"))
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)
import torch, torchvision, numpy, scipy, yaml
if not torch.cuda.is_available():
    raise RuntimeError("Esta receita do notebook exige CUDA; no Kaggle, habilite Accelerator = GPU.")
AMBIENTE = {
    "commit": COMMIT_CODIGO, "python": sys.version, "executavel": sys.executable,
    "torch": torch.__version__, "torchvision": torchvision.__version__,
    "numpy": numpy.__version__, "scipy": scipy.__version__, "pyyaml": yaml.__version__,
    "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0), "threads": 4, "workers": 2,
    "pacotes": subprocess.run([sys.executable, "-m", "pip", "freeze"],
                              capture_output=True, text=True, check=True).stdout.splitlines(),
}
print("torch", torch.__version__, "| GPU:", AMBIENTE["gpu"])

## 2. Insumos: MINDS para treinar e o backbone aprovado

Anexe datasets **privados**:
- Pacote `landmarks-minds.tar.gz` com `landmarks/` (ou pasta direta), contendo exatamente **800 clipes MINDS: oito pessoas, vinte sinais, reps 01–05**. Os arrays devem ser float32 finitos `(T>=3,57,3)`. Não anexar uma mistura MINDS + V-LIBRASIL: a pasta local histórica contém 30 clipes V adicionais que devem permanecer preservados fora deste pacote.
- Pacote de backbone com `backbone_gcn.pt`, ou arquivo equivalente explicitamente indicado. O hash é conferido antes do uso; não usar WLASL/negativos extras.

A preparação não altera os insumos: instala em destino exclusivo e só reutiliza uma preparação com identidade e bytes iguais. Se houver ambiguidade, preencher `ORIGEM_MINDS` e/ou `BACKBONE_EXPLICITO`. Opcionalmente fornecer `MANIFESTO_REFERENCIA`, inventário privado validado na origem. Hashes locais não certificam a extração histórica.

In [ ]:
import entrada_pretreino as entrada
import entrada_final

HASH_BACKBONE_APROVADO = "7a6e997c5830139162b32bc9b37a48e6eb5b8d6222836da87cb95e94ecf6baf5"
RAIZ_INPUT = pathlib.Path("/kaggle/input") if EM_KAGGLE else BASE
DESTINO = BASE / "dados-treino-final-v1"
# Se houver ambiguidade local/Kaggle, declarar caminhos explicitamente aqui.
ORIGEM_MINDS = None
BACKBONE_EXPLICITO = None
# Opcional: inventário privado validado na origem para conferir os mesmos bytes.
MANIFESTO_REFERENCIA = None

origem_minds = entrada.localizar(RAIZ_INPUT, "minds", ORIGEM_MINDS)
MINDS = entrada_final.preparar_minds(
    origem_minds, DESTINO / "minds", manifesto_referencia=MANIFESTO_REFERENCIA)
INVENTARIO_MINDS = entrada_final.validar_minds(MINDS)
print("MINDS completo:", INVENTARIO_MINDS["n_clipes"], "clipes |",
      INVENTARIO_MINDS["corpus_sha256"])

# Backbone em arquivo ou pacote; nunca extrair sobre uma pasta já usada.
if BACKBONE_EXPLICITO is not None:
    candidatos = [pathlib.Path(BACKBONE_EXPLICITO).resolve()]
else:
    candidatos = sorted(set(RAIZ_INPUT.rglob("backbone_gcn.pt")))
if len(candidatos) == 1:
    BACKBONE = candidatos[0]
elif candidatos:
    raise RuntimeError("Mais de um backbone: defina BACKBONE_EXPLICITO, sem escolha automática.")
else:
    tars = sorted(p for p in RAIZ_INPUT.rglob("*.tar.gz") if "backbone" in p.name)
    if len(tars) != 1:
        raise RuntimeError("Anexe exatamente um pacote de backbone ou defina BACKBONE_EXPLICITO.")
    DESTINO.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".backbone-", dir=DESTINO) as tmp:
        with tarfile.open(tars[0], "r:gz") as tar:
            membros = tar.getmembers()
            if any(not (m.isdir() or m.isfile()) for m in membros):
                raise RuntimeError("Pacote do backbone contém links ou arquivos irregulares.")
            tar.extractall(tmp, filter="data")
        achados = list(pathlib.Path(tmp).rglob("backbone_gcn.pt"))
        if len(achados) != 1:
            raise RuntimeError("Pacote não contém exatamente um backbone_gcn.pt.")
        bruto = achados[0].read_bytes()
        if entrada.pv.hash_arquivo(achados[0]) != HASH_BACKBONE_APROVADO:
            raise RuntimeError("Backbone do pacote não é o aprovado.")
        BACKBONE = DESTINO / f"backbone-{HASH_BACKBONE_APROVADO}.pt"
        if BACKBONE.exists():
            if BACKBONE.is_symlink() or BACKBONE.read_bytes() != bruto:
                raise RuntimeError("Backbone preparado foi alterado; nada será sobrescrito.")
        else:
            with BACKBONE.open("xb") as arquivo:
                arquivo.write(bruto)

if BACKBONE.is_symlink() or not BACKBONE.is_file():
    raise RuntimeError("Backbone deve ser arquivo regular.")
hash_real = entrada.pv.hash_arquivo(BACKBONE)
if hash_real != HASH_BACKBONE_APROVADO:
    raise RuntimeError(f"Backbone com hash {hash_real}; não usar.")
print("Backbone verificado:", BACKBONE, "|", hash_real)

## 3. Selftest — obrigatório antes do treino longo


In [ ]:
def executar(script, argumentos, log):
    comando = [sys.executable, "-u", script, *argumentos]
    print("Executando:", " ".join(comando), flush=True)
    with (EXP / log).open("a", encoding="utf-8") as arquivo_log:
        with subprocess.Popen(comando, cwd=TREINO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as processo:
            for linha in processo.stdout:
                print(linha, end="", flush=True)
                arquivo_log.write(linha)
                arquivo_log.flush()
            if processo.wait():
                raise subprocess.CalledProcessError(processo.returncode, comando)

NOME_EXPERIMENTO = "final-s20260917-v1"
EXP = BASE / "experimentos-privados" / NOME_EXPERIMENTO
EXP.mkdir(parents=True, exist_ok=True)

def registrar(nome, valor):
    caminho = EXP / nome
    texto = json.dumps(valor, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n"
    if caminho.exists():
        if caminho.is_symlink() or caminho.read_text(encoding="utf-8") != texto:
            raise RuntimeError(f"Registro {nome} diverge; use outro experimento, sem sobrescrever.")
    else:
        with caminho.open("x", encoding="utf-8") as f:
            f.write(texto)
    return caminho

registrar("ambiente.json", AMBIENTE)
INVENTARIO_CAMINHO = registrar("inventario-minds.json", INVENTARIO_MINDS)
registrar("inicializacao.json", {"backbone_sha256": hash_real, "commit": COMMIT_CODIGO})
codigo_final.conferir_codigo(REPO, COMMIT_APROVADO)
executar("test_entrada_final.py", [], "preflight.log")
executar("test_codigo_final.py", [], "preflight.log")
executar("test_politica_final.py", [], "preflight.log")
executar("selftest.py", [], "selftest.log")
print("Testes passaram. Saídas privadas:", EXP)

## 4. O treino final

`--politica-final ultima` exige `--semente` explícita e recusa saída não-vazia — não sobrescreve uma tentativa anterior por engano. Sem avaliação durante o treino: perda/acurácia no log são diagnóstico, não seleção.


In [ ]:
# Revalidar imediatamente antes da chamada longa, inclusive se células foram
# executadas fora de ordem. O próprio CLI repete a guarda antes de criar o modelo.
codigo_final.conferir_codigo(REPO, COMMIT_APROVADO)
entrada_final.validar_minds(MINDS, manifesto_referencia=INVENTARIO_CAMINHO)
if entrada.pv.hash_arquivo(BACKBONE) != HASH_BACKBONE_APROVADO:
    raise RuntimeError("Backbone mudou após o preflight.")
SAIDA_FINAL = EXP / "modelo_final"
FINAL_ARGS = [
    "--arquitetura", "gcn", "--ossos", "--com-z", "--z-recentrado",
    "--kernel-temporal", "9", "--fontes", "minds", "--landmarks", str(MINDS),
    "--inventario-final", str(INVENTARIO_CAMINHO), "--inicializar", str(BACKBONE),
    "--epocas", "120", "--lr", "1e-3", "--wd", "1e-4", "--batch", "64",
    "--agendador", "cosseno",
    "--final", "--politica-final", "ultima", "--semente", "20260917",
    "--dispositivo", "cuda", "--threads", "4", "--workers", "2", "--saida", str(SAIDA_FINAL),
]
registrar("comando-final.json", FINAL_ARGS)
executar("treinar.py", FINAL_ARGS, "treino-final.log")

destino = SAIDA_FINAL / "modelo_final.pt"
if not destino.is_file():
    raise RuntimeError("treinar.py terminou sem gerar modelo_final.pt.")
# Somente checkpoint próprio, gerado nesta run. Não carregar pickle de terceiros.
checkpoint = torch.load(destino, map_location="cpu", weights_only=False)
meta = checkpoint["meta"]
if (meta["epoca_salva"] != 120 or meta["politica_selecao"] != "ultima"
        or meta["avaliacao_independente"] is not False
        or meta["backbone_sha256"] != HASH_BACKBONE_APROVADO
        or checkpoint["rotulos"] != INVENTARIO_MINDS["rotulos"]
        or sorted(meta["pessoas"]) != INVENTARIO_MINDS["pessoas"]
        or meta["proveniencia"]["codigo"]["commit"] != COMMIT_APROVADO
        or any(not torch.isfinite(v).all() for v in checkpoint["state_dict"].values())):
    raise RuntimeError("Checkpoint não corresponde à política final; não exportar.")
registrar("checkpoint-final.json", {"sha256": entrada.pv.hash_arquivo(destino),
                                   "epoca": 120, "aprovado_entrega": False})
del checkpoint
print("Checkpoint final:", destino, "|", entrada.pv.hash_arquivo(destino))

## 5. Backup privado


In [ ]:
def empacotar():
    arquivo = EXP.parent / f"{EXP.name}.tar.gz"
    parcial = arquivo.with_suffix(".parcial")
    with tarfile.open(parcial, "w:gz") as tar:
        tar.add(EXP, arcname=EXP.name)
    parcial.replace(arquivo)
    return arquivo

arquivo = empacotar()
print("Arquivo privado:", arquivo, "|", arquivo.stat().st_size, "bytes")
print("Kaggle: Save Version e mantenha notebook, datasets e outputs PRIVADOS.")
